In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

In [ ]:
df = pd.read_csv('D:\ML-assignment\data\kidney_disease.csv')

print("Dataset shape:", df.shape)
df.head()

In [ ]:
print("=== Data Types & Non-null Counts ===")
df.info()

print("\n=== Missing Values Per Column ===")
print(df.isnull().sum())

df['classification'] = df['classification'].str.strip().str.lower()
print("\n=== Target Column Distribution ===")
print(df['classification'].value_counts())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------
# 1. CKD vs Not CKD Count Plot
# -----------------------------
plt.figure(figsize=(6,4))

ax = df['classification'].value_counts().plot(
    kind='bar',
    color=['#1D9E75', '#D85A30'],
    edgecolor='black'
)

plt.title('CKD vs Not CKD — Patient Count', fontsize=12, fontweight='bold')
plt.xlabel('Diagnosis', fontsize=10)
plt.ylabel('Number of Patients', fontsize=10)
plt.xticks(rotation=0)

# Add value labels on bars
for index, value in enumerate(df['classification'].value_counts()):
    plt.text(index, value + 2, str(value), ha='center', fontsize=10)

plt.tight_layout()
plt.show()


# ----------------------------------------
# 2. Feature Distributions by Class (KDE)
# ----------------------------------------
key_features = ['hemo', 'sc', 'bgr', 'bu', 'sod', 'pot']

plt.figure(figsize=(14,10))

for i, feat in enumerate(key_features):
    plt.subplot(3, 2, i+1)

    sns.kdeplot(X_imputed[feat][y == 1],
                label='CKD',
                fill=True,
                color='#2196F3')

    sns.kdeplot(X_imputed[feat][y == 0],
                label='Not CKD',
                fill=True,
                color='#FF5722')

    plt.title(f'{feat} Distribution by Class')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Fix column names & replace '?' values
df.columns = df.columns.str.strip()

df.replace('?', np.nan, inplace=True)

# Convert numeric columns
numeric_cols = ['age','bp','sg','al','su','bgr','bu',
                'sc','sod','pot','hemo','pcv','wc','rc']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check results
print("Data types after fix:")
print(df.dtypes)

print("\nMissing values now:")
print(df.isnull().sum())

In [ ]:
# Encode categorical (text) columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [c for c in categorical_cols if c != 'classification']

print("Columns to encode:", categorical_cols)

# Fill missing values before encoding (important!)
for col in categorical_cols:
    df[col].fillna('missing', inplace=True)

# Encode each column separately
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Clean and encode target column
df['classification'] = df['classification'].str.strip().str.lower()
df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})

print("\nEncoding complete!")
print("Target value counts:", df['classification'].value_counts().to_dict())

In [ ]:
# Separate features and target
X = df.drop('classification', axis=1)
y = df['classification']

print("X shape:", X.shape)
print("y shape:", y.shape)

# Impute missing values (mean strategy for numeric data)
imputer = SimpleImputer(strategy='mean')

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

# Reset index (good practice)
X_imputed = X_imputed.reset_index(drop=True)
y = y.reset_index(drop=True)

print("\nMissing values after imputation:",
      X_imputed.isnull().sum().sum())

In [ ]:
# Split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Impute (fit ONLY on training)
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Scale (fit ONLY on training)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

# 5-fold cross-validation — tests across 5 different splits
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv)

print("Model training complete!")
print(f"\nCross-Validation Accuracy across 5 folds:")
for i, score in enumerate(cv_scores):
    print(f"  Fold {i+1}: {score*100:.2f}%")
print(f"\nMean CV Accuracy : {cv_scores.mean()*100:.2f}%")
print(f"Std Deviation    : {cv_scores.std()*100:.2f}%")


In [ ]:
# Predict on test data
y_pred = model.predict(X_test)

# Predict probabilities (for ROC-AUC later)
y_prob = model.predict_proba(X_test)[:, 1]

# Display sample results
print("First 10 predictions:", y_pred[:10])
print("First 10 actual labels:", y_test.iloc[:10].values)

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Not CKD', 'CKD']
))

roc = roc_auc_score(y_test, y_prob)
print(f"\nROC-AUC Score: {roc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:


# 1️⃣ Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc * 100:.2f}%\n")

# 2️⃣ Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not CKD', 'CKD']))

# 3️⃣ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not CKD', 'CKD'], yticklabels=['Not CKD', 'CKD'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()

# 4️⃣ ROC Curve
roc_auc = roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0,1], [0,1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid()
plt.show()

In [ ]:
# Feature names
feature_names = X.columns

# Model coefficients
coefficients = model.coef_[0]

# Create DataFrame of feature importance
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients,
    'Abs': np.abs(coefficients)
}).sort_values('Abs', ascending=False)

# Top 10 features
print("Top 10 most important features:")
print(coef_df[['Feature', 'Coefficient']].head(10).to_string(index=False))

# Plot Top 10
plt.figure(figsize=(10,6))

top10 = coef_df.head(10)

colors = ['#D85A30' if x > 0 else '#534AB7' for x in top10['Coefficient']]

plt.barh(top10['Feature'], top10['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)

plt.title('Top 10 Feature Coefficients — Logistic Regression')
plt.xlabel('Coefficient Value')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Positive risk factors
print("\nPositive risk factors (increase CKD risk):")
print(coef_df[coef_df['Coefficient'] > 0][['Feature','Coefficient']].head().to_string(index=False))

# Negative risk factors
print("\nProtective factors (reduce CKD risk):")
print(coef_df[coef_df['Coefficient'] < 0][['Feature','Coefficient']].head().to_string(index=False))

In [ ]:


# Compute confusion matrix once
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("=" * 50)
print("  LOGISTIC REGRESSION — RESULTS SUMMARY")
print("=" * 50)

print(f"  Dataset       : Chronic Kidney Disease")
print(f"  Total samples : {len(df)}")
print(f"  Train/Test    : 80% / 20%")
print(f"  Features used : {X_train.shape[1]}")
print("-" * 50)

print(f"  Accuracy      : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"  ROC-AUC       : {roc_auc_score(y_test, y_prob):.4f}")

print("-" * 50)
print(f"  True Positives  (CKD correctly detected)   : {tp}")
print(f"  True Negatives  (Healthy correctly cleared) : {tn}")
print(f"  False Positives (Healthy wrongly flagged)   : {fp}")
print(f"  False Negatives (CKD missed)                : {fn}")

print("=" * 50)